# Gradient Cobra

## Combine Classifier

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(
    f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features\n"
    f"Test set: {X_test.shape[0]} samples, {X_test.shape[1]} features"
)

Training set: 455 samples, 30 features
Test set: 114 samples, 30 features


In [2]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from cobra.combine_classifier import CombineClassifier

model = CombineClassifier(
    splitter="holdout",
    distance="hamming",
    kernel="indicator",
    aggregator="majority_vote",
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [3]:
preds = model.predict(X_test)
accuracy = (preds == y_test).mean()
print(f"Test set accuracy: {accuracy:.4f}")

Test set accuracy: 0.8860


In [4]:
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

baselines = {
    "decision_tree": DecisionTreeClassifier(random_state=42),
    "logistic_regression": LogisticRegression(max_iter=5000),
    "random_forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "gradient_boosting": GradientBoostingClassifier(),
    "knn": KNeighborsClassifier(),
    "svm": SVC()
}

results = []

for name, model in baselines.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    results.append(("baseline_" + name, acc))

model = CombineClassifier(
    estimators=baselines.keys(),
    splitter="holdout",
    distance="hamming",
    kernel="indicator",
    aggregator="majority_vote",
    random_state=42
)
model.fit(X_train, y_train)
preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)
results.append(("combine_classifier", acc))

for name, acc in results:
    print(f"{name}: {acc:.4f}")


baseline_decision_tree: 0.9474
baseline_logistic_regression: 0.9561
baseline_random_forest: 0.9649
baseline_gradient_boosting: 0.9561
baseline_knn: 0.9561
baseline_svm: 0.9474
combine_classifier: 0.9561


In [5]:
from cobra.core.estimators.base import BaseEstimator, EstimatorFactory
EstimatorFactory.available()

['decision_tree',
 'dummy_mean',
 'gradient_boosting',
 'knn',
 'lasso',
 'linear',
 'logistic_regression',
 'mean_regressor',
 'random_forest',
 'ridge',
 'svm']

# GradientCobra

In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
X, y = fetch_california_housing(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(
    f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features\n"
    f"Test set: {X_test.shape[0]} samples, {X_test.shape[1]} features"
)

Training set: 16512 samples, 8 features
Test set: 4128 samples, 8 features


In [2]:
from cobra.gradientcobra import GradientCOBRA
import numpy as np

model = GradientCOBRA(
    splitter="holdout",
    distance="euclidean",
    kernel="rbf",
    aggregator="weighted_mean",
    optimizer="grad",
    random_state=42
)
model.fit(X_train, y_train)

TypeError: BaseKernel.__init__() missing 1 required positional argument: 'alpha'

In [ ]:
model.optimization_outputs_

{'method': 'grad',
 'bandwidth': array([0.44009208]),
 'risk': 0.3080310847991036,
 'histories': [{'iteration': 0,
   'x': array([0.99451276]),
   'score': 0.33910292775381823,
   'best_score': 0.33910292775381823,
   'gradient': array([0.05487242])},
  {'iteration': 1,
   'x': array([0.98902451]),
   'score': 0.33880169075624217,
   'best_score': 0.33880169075624217,
   'gradient': array([0.05488244])},
  {'iteration': 2,
   'x': array([0.98353521]),
   'score': 0.33850033642987454,
   'best_score': 0.33850033642987454,
   'gradient': array([0.054893])},
  {'iteration': 3,
   'x': array([0.97804481]),
   'score': 0.3381988593436668,
   'best_score': 0.3381988593436668,
   'gradient': array([0.05490407])},
  {'iteration': 4,
   'x': array([0.97255325]),
   'score': 0.33789725445535024,
   'best_score': 0.33789725445535024,
   'gradient': array([0.05491559])},
  {'iteration': 5,
   'x': array([0.96706049]),
   'score': 0.3375955171138183,
   'best_score': 0.3375955171138183,
   'gradien

In [ ]:
import numpy as np
preds = model.predict(X_test)
mse = np.mean((preds - y_test) ** 2)
print(f"Test set MSE: {mse:.4f}")

Test set MSE: 0.2899


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

baselines = {
    "linear": LinearRegression(),
    "ridge": Ridge(),
    "random_forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "svm": SVR()
}
results = []
for name, model in baselines.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mse = mean_squared_error(y_test, preds)
    results.append(("baseline_" + name, mse))

model = GradientCOBRA(
    estimators=baselines.keys(),
    splitter="holdout",
    distance="euclidean",
    kernel="rbf",
    aggregator="weighted_mean",
    random_state=42
)
model.fit(X_train, y_train)
preds = model.predict(X_test)
mse = mean_squared_error(y_test, preds)
results.append(("gradient_cobra", mse))

for name, mse in results:
    print(f"{name}: {mse:.4f}")

baseline_linear: 0.5559
baseline_ridge: 0.5558
baseline_random_forest: 0.2539
baseline_svm: 1.3320
gradient_cobra: 0.2893


# MixCobra

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
X, y = fetch_california_housing(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(
    f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features\n"
    f"Test set: {X_test.shape[0]} samples, {X_test.shape[1]} features"
)

Training set: 16512 samples, 8 features
Test set: 4128 samples, 8 features


In [ ]:
from cobra.mixcobra import MixCOBRARegressor
model = MixCOBRARegressor(
    splitter="holdout",
    distance="euclidean",
    kernel="rbf",
    aggregator="weighted_mean",
    loss="mse",
    optimizer="grid",
    random_state=42
)
model.fit(X_train, y_train)

Grid Search: 100%|██████████| 25/25 [00:30<00:00,  1.21s/it]


,estimators,None
,estimators_params,None
,splitter,'holdout'
,splitter_params,None
,distance,'euclidean'
,distance_params,None
,kernel,'rbf'
,kernel_params,None
,aggregator,'weighted_mean'
,aggregator_params,None
,loss,'mse'


In [ ]:
from sklearn.metrics import mean_squared_error
preds = model.predict(X_test)
mse = mean_squared_error(y_test, preds)
print(f"Test set MSE: {mse:.4f}")

Test set MSE: 1.2859


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error

from cobra.mixcobra import MixCOBRARegressor

# ======================
# SINGLE MODELS
# ======================
single_models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42),
    "SVR": SVR(C=5.0, epsilon=0.1)
}

results = {}

# train single models
for name, model in single_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results[name] = mean_squared_error(y_test, pred)

# ======================
# MIXCOBRA
# ======================
mix = MixCOBRARegressor(
    kernel="rbf",
    distance="euclidean",
    aggregator="weighted_mean",
    alpha_grid=np.linspace(0.1, 3.0, 10),
    beta_grid=np.linspace(0.1, 3.0, 10),
    random_state=42
)

mix.fit(X_train, y_train)
mix_pred = mix.predict(X_test)

results["MixCOBRA"] = mean_squared_error(y_test, mix_pred)

# ======================
# RESULTS
# ======================
print("\n===== MSE COMPARISON =====\n")

for name, score in sorted(results.items(), key=lambda x: x[1]):
    print(f"{name:20s} : {score:.4f}")


===== MSE COMPARISON =====

RandomForest         : 0.2537
MixCOBRA             : 0.4114
Ridge                : 0.5558
LinearRegression     : 0.5559
SVR                  : 1.2197
